In [ ]:
import os
import sys
%pip install psycopg2 networkx matplotlib numpy
import psycopg2
from db_setup import get_connection
import networkx as nx
import matplotlib.pyplot as plt
import pylab as pl
import numpy as np
import itertools
from collections import Counter
%pip install --config-settings="--global-option=build_ext"  --config-settings="--global-option=-IC:\Program Files\Graphviz\include"  --config-settings="--global-option=-LC:\Program Files\Graphviz\lib" pygraphviz
import pygraphviz
from networkx.drawing.nx_agraph import graphviz_layout
from operator import itemgetter
import math
from datetime import datetime


In [ ]:
db = get_connection()

In [ ]:
queries = [
    # The entire community - 0
    (
        """
        SELECT id
        FROM publications
        WHERE year between 2011 AND 2020
        """,
        "systems-community"
    ),

    # Workflow communities - 1
    (
        """
        SELECT id
        FROM publications
        WHERE year between 2011 AND 2020
        AND (lower(title) LIKE '%workflow%' OR lower(abstract) LIKE '%workflow%')
        AND (lower(title) LIKE '%schedul%' OR lower(abstract) LIKE '%schedul%')
        """,
        "workflow-community"
    ),

    # Workflow formalisms - 2
    (
        """
        SELECT id
        FROM publications
        WHERE year between 2011 AND 2020
        AND (lower(title) LIKE '%workflow%' OR lower(abstract) LIKE '%workflow%')
        AND (
            (lower(title) LIKE '%formalism%' OR lower(abstract) LIKE '%formalism%')
            OR (lower(title) LIKE '%language%' OR lower(abstract) LIKE '%language%')
        )
        """,
        "workflow-formalism-community"
    ),

    # Workflow allocation - 3
    (
        """
        SELECT id
        FROM publications
        WHERE year between 2011 AND 2020
        AND (lower(title) LIKE '%workflow%' OR lower(abstract) LIKE '%workflow%')
        AND (
            (lower(title) LIKE '%allocat%' OR lower(abstract) LIKE '%allocat%')
            OR (lower(title) LIKE '%schedul%' OR lower(abstract) LIKE '%schedul%')
            OR (lower(title) LIKE '%plan%' OR lower(abstract) LIKE '%plan%')
        )
        """,
        "workflow-allocation-community"
    ),

    # Resource provisioning communities - 4
    (
        """
        SELECT id
        FROM publications
        WHERE year between 2011 AND 2020
        AND (lower(title) LIKE '%workflow%' OR lower(abstract) LIKE '%workflow%')
        AND (
            lower(title) LIKE '%provision%' OR lower(abstract) LIKE '%provision%'
            OR lower(title) LIKE '%autoscal%' OR lower(abstract) LIKE '%autoscal%'
        )
        """,
        "resource-provisioning-community"
    ),

    # applications and services - 5
    (
        """
        SELECT id
        FROM publications
        WHERE year between 2011 AND 2020
        AND (lower(title) LIKE '%cloud%' OR lower(abstract) LIKE '%cloud%')
        AND (lower(title) LIKE '%service%' OR lower(abstract) LIKE '%service%')
        """,
        "applications-and-services-community"
    ),
]

In [ ]:
def generate_graph_for_publication_query(search_query):
    query = """
    SELECT a.id, a.name, app.paper_id 
    FROM authors a JOIN author_paper_pairs app ON CAST(a.id AS INTEGER) = app.author_id
    WHERE app.paper_id IN (
        {}
    ) ORDER BY app.paper_id""".format(search_query)

    with db:
        with db.cursor() as cursor:
            cursor.execute(query)
            author_list = cursor.fetchall()

    g = nx.Graph()
    author_set = set()
    author_id_to_label = dict()
    author_relation_set = set()
    author_list_len = len(author_list)
    i = 0

    num_relations = 0
    num_articles = 0
    while i < author_list_len:
        author_id = author_list[i][0]
        author_name = author_list[i][1]
        paper_id = author_list[i][2]
        num_articles += 1

        if author_id not in author_set:
            author_set.add(author_id)
            author_id_to_label[author_id] = author_name
            g.add_node(author_id, label=author_name)

        # Make sure to loop from i+1, to avoid self-loops yet still capture the many-to-many relationship
        for j in range(i + 1, len(author_list)):
            co_author_id = author_list[j][0]
            co_author_name = author_list[j][1]
            co_paper_id = author_list[j][2]

            # Check if we reached a new article
            if co_paper_id != paper_id:
                break

            if co_author_id not in author_set:
                author_set.add(co_author_id)
                author_id_to_label[co_author_id] = co_author_name
                g.add_node(co_author_id, label=co_author_name)

            num_relations += 1
            # Put a relation of (smallest id, larger id) to only add an edge once later.
            # Doing this ensures we do not add a relationship more than once.
            if (min(author_id, co_author_id),
                max(author_id, co_author_id)) not in author_relation_set:
                author_relation_set.add((min(author_id, co_author_id),
                                         max(author_id, co_author_id)))
                g.add_edge(min(author_id, co_author_id),
                           max(author_id, co_author_id))
        i += 1

    return g

In [ ]:
def visualize_cliques_numbers_in_graph(g, community_string):
    minimum_clique_size = 1
    cliques = list(
        x for x in nx.find_cliques(g) if len(x) >= minimum_clique_size)

    cliques_counter = Counter([len(c) for c in cliques])
    print(
        "Number of cliques with minimum size of {}: {} community string: {}".format(
            minimum_clique_size, len(cliques), community_string))
    print("Largest clique: {}".format(max(cliques_counter.keys())))

    # for i in range(len(cliques)):
    #     if len(cliques[i]) < 100:
    #         continue
    #     print("Clique {} with {} members:".format(i+1, len(cliques[i])))
    #     print(", ".join([author_id_to_label[author_id] for author_id in cliques[i]]))
    #     print()

    # Create a bargraph on the distribution of cliques
    fig, ax = pl.subplots(figsize=(16, 16))
    ax.bar(cliques_counter.keys(), cliques_counter.values())

    if max(cliques_counter.keys()) < 50:  # We can probaby visualize this without text above the bars overlapping
        max_y_value = ax.get_ylim()[1]
    #         text_margin = max_y_value / 100
    #         for key, value in cliques_counter.items():
    #             ax.text(key, value + text_margin, str(value), color='black', ha='center', fontweight='normal', fontsize=28)
    else:  # Zooms in on the lower axis to show the small 
        # Inner Plot
        inner_ax = fig.add_axes([0.21, 0.2, 0.67, 0.4])  # x, y, width, height
        inner_ax.bar(cliques_counter.keys(), cliques_counter.values())

        ylim = 10
        text_margin = ylim / 100

        inner_ax.set(title='Zoom In', xlim=(0, 300), ylim=(0, ylim))
        #         for key, value in cliques_counter.items():
        #             if value < ylim:
        #                 inner_ax.text(key, value + text_margin, str(value), color='black', ha='center', fontweight='normal', fontsize=26)

        # Set the fontsize of the inner plot
        for item in ([inner_ax.title, inner_ax.xaxis.label,
                      inner_ax.yaxis.label] + inner_ax.get_xticklabels() + inner_ax.get_yticklabels()):
            item.set_fontsize(26)

        # We manually set the visibility of the lines from the original graph to the zoomed in part
        # as MatplotLib is choosing slightly akward lines itself.
        patch, connector_lines = ax.indicate_inset_zoom(inner_ax)
        connector_lines[0].set_visible(True)
        connector_lines[1].set_visible(False)
        connector_lines[2].set_visible(True)
        connector_lines[3].set_visible(False)

    ax.set_xlabel("Clique size", fontsize=40)
    ax.set_ylabel("Number of cliques", fontsize=40)

    for item in ([ax.title, ax.xaxis.label,
                  ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels()):
        item.set_fontsize(34)

    date_time = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
    fig.savefig('cliques_{}_{}.pdf'.format(community_string, date_time))

In [ ]:
search_query, community_string = queries[4]
graph = generate_graph_for_publication_query(search_query)
pos = nx.spring_layout(graph, scale=3, k=0.2)

nx.draw(graph, pos, edge_color='gray', node_size=2, font_size=12)


In [ ]:
def generate_scatterplot_average_citation_count_vs_clique_size(g,
                                                               community_string):
    minimum_clique_size = 1
    cliques = list(
        x for x in nx.find_cliques(g) if len(x) >= minimum_clique_size)

    avg_citation_count_vs_clique_size = []
    max_citation_count_vs_clique_size = []

    for clique in cliques:
        author_citation_count = []
        for author_id in clique:
            query = """
                SELECT COALESCE(SUM(n_citations), 0)
                FROM publications p JOIN author_paper_pairs app ON p.id = app.paper_id
                WHERE app.author_id = %s
                AND n_citations >= 0
            """
            with db:
                with db.cursor() as cursor:
                    cursor.execute(query, [author_id])
                    author_sum = int(cursor.fetchone()[0])
                    author_citation_count.append(author_sum)

        average_citation_count_per_author = np.average(author_citation_count)
        max_citation_count_per_author = np.max(author_citation_count)
        avg_citation_count_vs_clique_size.append(
            (average_citation_count_per_author, len(clique)))
        max_citation_count_vs_clique_size.append(
            (max_citation_count_per_author, len(clique)))

    # Generate the scatterplot
    print(avg_citation_count_vs_clique_size)
    # Set the limit to the next integer given the exponent, so 3.5*10^3 will become 4*10^3, etc.
    max_x_value = max(max_citation_count_vs_clique_size, key=itemgetter(0))[0]
    lower_exp = math.floor(math.log10(max_x_value))
    x_limit = math.ceil(max_x_value / 10 ** lower_exp) * (10 ** lower_exp)

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(32, 12))
    ax = axes[1]
    ax.ticklabel_format(style='sci', axis='x', scilimits=(0, 0),
                        useMathText=True)
    ax.scatter([d for d, _ in max_citation_count_vs_clique_size],
               [d for _, d in max_citation_count_vs_clique_size], s=200)
    ax.set_xlabel("Max Citation Count of Author in Clique", fontsize=60,
                  labelpad=60)
    ax.set_ylabel("Clique size", fontsize=60)
    ax.set_xlim(0, x_limit)

    for item in ([ax.title, ax.xaxis.label,
                  ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels()):
        item.set_fontsize(54)
    ax.xaxis.get_offset_text().set_fontsize(44)

    ax = axes[0]
    ax.ticklabel_format(style='sci', axis='x', scilimits=(0, 0),
                        useMathText=True)
    ax.scatter([d for d, _ in avg_citation_count_vs_clique_size],
               [d for _, d in avg_citation_count_vs_clique_size], s=200)
    ax.set_xlabel("Average Citation Count of Authors in Clique", fontsize=60,
                  labelpad=60)
    ax.set_ylabel("Clique size", fontsize=60)
    ax.set_xlim(0, x_limit)

    for item in ([ax.title, ax.xaxis.label,
                  ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels()):
        item.set_fontsize(54)
    ax.xaxis.get_offset_text().set_fontsize(44)

    plt.tight_layout()
    date_time = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
    fig.savefig(
        'cliques_size_vs_author_citation_{}_{}.pdf'.format(community_string,
                                                           date_time),
        pad_inches=0)

In [ ]:
generate_scatterplot_average_citation_count_vs_clique_size(graph,
                                                           community_string)
print(community_string)

In [ ]:
graphs_per_community = dict()
for search_query, community_string in queries:
    graph = generate_graph_for_publication_query(search_query)
    graphs_per_community[community_string] = graph

In [ ]:
for community_string, graph in graphs_per_community.items():
    generate_scatterplot_average_citation_count_vs_clique_size(graph,
                                                               community_string)